## 1. Imports

In [ ]:
# >>> pRNA figure routing (added by the pRNA update; keep as the first code cell)
# Every figure is shown inline. Every file written with savefig (png, pdf, svg) is redirected to
# figures_pdf/<notebook name>/ next to this notebook; sub-folders of the original path are preserved.
import os as _fig_os
from pathlib import Path as _FigPath
import matplotlib as _fig_mpl
import matplotlib.pyplot as _fig_plt
from matplotlib.figure import Figure as _FigFigure

NOTEBOOK_NAME = "benchmark_scTranslator_overlap_tmt"
FIG_DIR = (_FigPath.cwd() / "figures_pdf" / NOTEBOOK_NAME).resolve()
FIG_DIR.mkdir(parents=True, exist_ok=True)


def _route_figure_path(fname):
    p = _FigPath(_fig_os.fspath(fname))
    full = p if p.is_absolute() else (_FigPath.cwd() / p)
    try:
        full.resolve().relative_to(FIG_DIR)
        return fname                                  # already inside this notebook's figure folder
    except ValueError:
        pass
    try:
        rel = _FigPath(_fig_os.path.relpath(full.resolve(), _FigPath.cwd()))
    except ValueError:                                # e.g. another drive on Windows
        rel = _FigPath(p.name)
    parts = [x for x in rel.parts if x not in ("..", ".", "figures_pdf", NOTEBOOK_NAME)]
    target = FIG_DIR.joinpath(*parts) if parts else FIG_DIR / p.name
    target.parent.mkdir(parents=True, exist_ok=True)
    return target


if not getattr(_FigFigure.savefig, "_prna_routed", False):
    _fig_orig_savefig = _FigFigure.savefig

    def _fig_routed_savefig(self, fname, *args, **kwargs):
        if isinstance(fname, (str, bytes, _FigPath)) or hasattr(fname, "__fspath__"):
            fname = _route_figure_path(fname)
        return _fig_orig_savefig(self, fname, *args, **kwargs)

    _fig_routed_savefig._prna_routed = True
    _FigFigure.savefig = _fig_routed_savefig

    _fig_orig_close = _fig_plt.close

    def _fig_close_and_show(fig=None):
        """Display figures that are closed before Jupyter could render them, then close."""
        import inspect as _fig_inspect
        _caller = _fig_inspect.currentframe().f_back
        _caller_mod = (_caller.f_globals.get("__name__", "") if _caller is not None else "")
        if _caller_mod.startswith(("matplotlib", "ipykernel")):   # backend housekeeping, not user code
            return _fig_orig_close(fig)
        if "inline" in str(_fig_mpl.get_backend()).lower():
            from IPython.display import display as _fig_display
            open_nums = set(_fig_plt.get_fignums())
            if fig is None:
                figs = [_fig_plt.gcf()] if open_nums else []
            elif isinstance(fig, str) and fig == "all":
                figs = [_fig_plt.figure(n) for n in sorted(open_nums)]
            elif isinstance(fig, _FigFigure):
                figs = [fig] if getattr(fig, "number", None) in open_nums else []
            elif isinstance(fig, int):
                figs = [_fig_plt.figure(fig)] if fig in open_nums else []
            else:
                figs = []
            for _f in figs:
                if not getattr(_f, "_prna_displayed", False):
                    _f._prna_displayed = True
                    _fig_display(_f)
        return _fig_orig_close(fig)

    _fig_plt.close = _fig_close_and_show

    _fig_orig_show = _fig_plt.show

    def _fig_show_and_mark(*args, **kwargs):
        for _n in _fig_plt.get_fignums():                    # shown now -> never re-displayed on close
            _fig_plt.figure(_n)._prna_displayed = True
        return _fig_orig_show(*args, **kwargs)

    _fig_plt.show = _fig_show_and_mark

print("Figures are saved to", FIG_DIR)

In [ ]:
import os
import sys
import pickle
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

## 2. Paths & Configuration

In [ ]:
# ── Absolute paths ──────────────────────────────────────────────────────────────
# ── Paths (portable: env vars, else <repo>/external/...) ─────────────────────────
from pathlib import Path as _Path
def _find_repo_root(start=_Path.cwd().resolve()):
    for p in [start, *start.parents]:
        if (p / '06. validation').is_dir() and (p / '01. pre-processing').is_dir():
            return p
    return start
REPO_ROOT = _find_repo_root()
SCTRANSLATOR_REPO = os.environ.get('SCTRANSLATOR_REPO', str(REPO_ROOT / 'external' / 'scTranslator' / 'scTranslator'))
CHECKPOINT_DIR    = os.environ.get('SCTRANSLATOR_CKPT_DIR', str(REPO_ROOT / 'external' / 'scTranslator' / 'checkpoints'))
VAL_DATA_DIR      = os.environ.get('VAL_DATA_DIR', str(REPO_ROOT / '06. validation' / 'val_data'))

def _torch_load(path, map_location):
    """Checkpoints are full pickled models: PyTorch>=2.6 needs weights_only=False (trusted files only)."""
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:  # PyTorch < 1.13 has no weights_only argument
        return torch.load(path, map_location=map_location)

# In a notebook __file__ is not defined, so we use the current working directory.
# Inputs are read from 06. validation/val_data (written by notebooks/2. comparison.ipynb).
SCRIPT_DIR = os.path.dirname(os.path.abspath('__file__'))  # notebook directory

# Add scTranslator code to path
sys.path.append(os.path.join(SCTRANSLATOR_REPO, 'code', 'model'))
from performer_enc_dec import scPerformerEncDec

print(f"SCRIPT_DIR:        {SCRIPT_DIR}")
print(f"SCTRANSLATOR_REPO: {SCTRANSLATOR_REPO}")
print(f"CHECKPOINT_DIR:    {CHECKPOINT_DIR}")
print(f"VAL_DATA_DIR:      {VAL_DATA_DIR}")

In [ ]:
CONFIG = {
    'rna_path':  os.path.join(VAL_DATA_DIR, 'TMT_transcriptomics.joblib'),
    'prot_path': os.path.join(VAL_DATA_DIR, 'TMT_proteomics.joblib'),
    'dic_path':  os.path.join(SCTRANSLATOR_REPO, 'code', 'model', 'ID_dic'),
    'checkpoints': {
        '2M':   os.path.join(CHECKPOINT_DIR, 'scTranslator_2M.pt'),
        '10K':  os.path.join(CHECKPOINT_DIR, 'scTranslator_10K.pt'),
        '160K': os.path.join(CHECKPOINT_DIR, 'scTranslator_160K.pt'),
    },
    'enc_max_seq_len': 20000,
    'dec_max_seq_len': 1000,   # Decoder limit
    'batch_size': 8,          
    'output_dir': os.path.join(SCRIPT_DIR, 'benchmark_results_OVERLAP_tmt'),
    # Overlap gene/protein lists to restrict features
    'overlap_genes_path': os.path.join(VAL_DATA_DIR, 'genes', 'overlap_ALL_4.txt'),
    'overlap_proteins_path': os.path.join(VAL_DATA_DIR, 'proteins', 'overlap_ALL_4.txt'),
}

# Quick sanity check – print which checkpoint files exist
for name, path in CONFIG['checkpoints'].items():
    status = 'FOUND' if os.path.exists(path) else 'MISSING'
    print(f"  [{status}] {name}: {path}")

## 3. ID Mapping Helpers

In [ ]:
def load_id_dictionaries(dic_path):
    """Load scTranslator ID mapping dictionaries."""
    with open(os.path.join(dic_path, 'hgs_to_EntrezID.pkl'), 'rb') as f:
        hgs_to_entrez = pickle.load(f)
    with open(os.path.join(dic_path, 'EntrezID_to_myID.pkl'), 'rb') as f:
        entrez_to_myid = pickle.load(f)
    return hgs_to_entrez, entrez_to_myid


def gene_symbol_to_myid(gene_symbol, hgs_to_entrez, entrez_to_myid):
    """Convert gene symbol to scTranslator ID."""
    entrez_id = hgs_to_entrez.get(gene_symbol)
    if entrez_id:
        my_id = entrez_to_myid.get(entrez_id)
        return int(my_id) if my_id else None
    return None


def map_genes_to_ids(gene_list, hgs_to_entrez, entrez_to_myid):
    """Map a list of gene symbols to scTranslator IDs."""
    mapped = []
    unmapped = []
    for gene in gene_list:
        my_id = gene_symbol_to_myid(gene, hgs_to_entrez, entrez_to_myid)
        if my_id is not None:
            mapped.append((gene, my_id))
        else:
            unmapped.append(gene)
    return mapped, unmapped

## 4. Normalization

In [ ]:
def cellwise_minmax(X, low=1e-8, high=1.0):
    """
    Paper's normalization: per-cell min-max scaling.
    Maps each cell's values to [low, high] range.
    """
    X = X.astype(np.float32)
    mins = X.min(axis=1, keepdims=True)
    maxs = X.max(axis=1, keepdims=True)
    denom = (maxs - mins)
    denom[denom == 0] = 1.0  # Avoid division by zero
    Xn = (X - mins) / denom
    Xn = Xn * (high - low) + low
    return Xn.astype(np.float32)

## 5. Data Preparation (NaN Handling + ID Mapping)

**NaN strategy (threshold-based):**
- Columns where **>50 %** of values across samples are NaN are **dropped**.
- Remaining NaN values are **filled with `1e-8`** (a small non-zero value; 0 is the special prediction token for scTranslator's decoder).

In [ ]:
def prepare_data(config):
    """
    Load and prepare RNA and protein data with ID mapping.

    NaN handling (threshold-based):
    - Columns (genes / proteins) where >50% of values across samples are NaN
      are dropped entirely.
    - Remaining NaN values are filled with 1e-8 (a small non-zero value,
      since 0 is the special prediction token for scTranslator's decoder).
    """
    NAN_THRESHOLD = 0.50  # drop column if more than 50% of samples are NaN

    print("Loading data...")
    rna_df = joblib.load(config['rna_path'])
    prot_df = joblib.load(config['prot_path'])

    print(f"RNA shape (raw):     {rna_df.shape}")
    print(f"Protein shape (raw): {prot_df.shape}")

    # ── Restrict to overlap gene/protein lists ──────────────────────────
    if config.get('overlap_genes_path') and os.path.exists(config['overlap_genes_path']):
        overlap_genes = pd.read_csv(config['overlap_genes_path'], header=None)[0].tolist()
        rna_keep = [g for g in rna_df.columns if g in set(overlap_genes)]
        rna_df = rna_df[rna_keep]
        print(f"Restricted RNA genes to overlap list: {len(overlap_genes)} allowed -> {len(rna_keep)} kept")
    if config.get('overlap_proteins_path') and os.path.exists(config['overlap_proteins_path']):
        overlap_proteins = pd.read_csv(config['overlap_proteins_path'], header=None)[0].tolist()
        prot_keep = [p for p in prot_df.columns if p in set(overlap_proteins)]
        prot_df = prot_df[prot_keep]
        print(f"Restricted proteins to overlap list:  {len(overlap_proteins)} allowed -> {len(prot_keep)} kept")

    rna_orig_cols  = len(rna_df.columns)
    prot_orig_cols = len(prot_df.columns)
    n_samples_rna  = len(rna_df)
    n_samples_prot = len(prot_df)

    print(f"RNA shape (after overlap filter):     {rna_df.shape}")
    print(f"Protein shape (after overlap filter): {prot_df.shape}")

    # ── NaN handling: drop if >50% NaN, else fill with 1e-8 ─────────────
    rna_nan_frac  = rna_df.isna().sum() / n_samples_rna
    prot_nan_frac = prot_df.isna().sum() / n_samples_prot

    rna_drop_cols  = rna_nan_frac[rna_nan_frac > NAN_THRESHOLD].index.tolist()
    prot_drop_cols = prot_nan_frac[prot_nan_frac > NAN_THRESHOLD].index.tolist()

    rna_fill_cols  = rna_nan_frac[(rna_nan_frac > 0) & (rna_nan_frac <= NAN_THRESHOLD)].index.tolist()
    prot_fill_cols = prot_nan_frac[(prot_nan_frac > 0) & (prot_nan_frac <= NAN_THRESHOLD)].index.tolist()

    # Drop columns exceeding the NaN threshold
    rna_df  = rna_df.drop(columns=rna_drop_cols)
    prot_df = prot_df.drop(columns=prot_drop_cols)

    # Fill remaining NaN values with 1e-8 (not 0, which is reserved)
    rna_df  = rna_df.fillna(1e-8)
    prot_df = prot_df.fillna(1e-8)

    print(f"\nNaN handling (threshold: >{NAN_THRESHOLD*100:.0f}% NaN -> drop, else fill with 1e-8):")
    print(f"  RNA genes dropped  (>{NAN_THRESHOLD*100:.0f}% NaN): {len(rna_drop_cols):,} / {rna_orig_cols} "
          f"({100*len(rna_drop_cols)/rna_orig_cols:.1f}%)")
    print(f"  RNA genes filled   (<=50% NaN):  {len(rna_fill_cols):,} / {rna_orig_cols} "
          f"({100*len(rna_fill_cols)/rna_orig_cols:.1f}%)")
    print(f"  Proteins dropped   (>{NAN_THRESHOLD*100:.0f}% NaN): {len(prot_drop_cols):,} / {prot_orig_cols} "
          f"({100*len(prot_drop_cols)/prot_orig_cols:.1f}%)")
    print(f"  Proteins filled    (<=50% NaN):  {len(prot_fill_cols):,} / {prot_orig_cols} "
          f"({100*len(prot_fill_cols)/prot_orig_cols:.1f}%)")
    print(f"  RNA shape after:     {rna_df.shape}")
    print(f"  Protein shape after: {prot_df.shape}")

    # ── ID mapping ──────────────────────────────────────────────────────
    hgs_to_entrez, entrez_to_myid = load_id_dictionaries(config['dic_path'])

    # Map RNA genes
    print("\nMapping RNA genes to scTranslator IDs...")
    rna_mapped, rna_unmapped = map_genes_to_ids(
        rna_df.columns.tolist(), hgs_to_entrez, entrez_to_myid
    )
    print(f"  RNA genes after NaN removal: {len(rna_df.columns)}")
    print(f"  Mapped to scTranslator:      {len(rna_mapped)} ({100*len(rna_mapped)/len(rna_df.columns):.1f}%)")
    print(f"  Unmapped (thrown out):        {len(rna_unmapped)} ({100*len(rna_unmapped)/len(rna_df.columns):.1f}%)")
    if rna_unmapped:
        print(f"  First 20 unmapped genes:     {rna_unmapped[:20]}")

    # Map protein genes
    print("\nMapping proteins to scTranslator IDs...")
    prot_mapped, prot_unmapped = map_genes_to_ids(
        prot_df.columns.tolist(), hgs_to_entrez, entrez_to_myid
    )
    print(f"  Proteins after NaN removal:  {len(prot_df.columns)}")
    print(f"  Mapped to scTranslator:      {len(prot_mapped)} ({100*len(prot_mapped)/len(prot_df.columns):.1f}%)")
    print(f"  Unmapped (thrown out):        {len(prot_unmapped)} ({100*len(prot_unmapped)/len(prot_df.columns):.1f}%)")
    if prot_unmapped:
        print(f"  Unmapped proteins:           {prot_unmapped}")

    # ── Summary of total attrition ──────────────────────────────────────
    print(f"\n  === Total gene / protein attrition ===")
    print(f"  RNA:  {rna_orig_cols} -> {len(rna_mapped)}  "
          f"(dropped {len(rna_drop_cols)} NaN + {len(rna_unmapped)} unmapped "
          f"= {rna_orig_cols - len(rna_mapped)} total removed, "
          f"{len(rna_fill_cols)} cols NaN-filled)")
    print(f"  Prot: {prot_orig_cols} -> {len(prot_mapped)}  "
          f"(dropped {len(prot_drop_cols)} NaN + {len(prot_unmapped)} unmapped "
          f"= {prot_orig_cols - len(prot_mapped)} total removed, "
          f"{len(prot_fill_cols)} cols NaN-filled)")

    # Extract mapped data
    rna_genes  = [g for g, _ in rna_mapped]
    rna_ids    = [i for _, i in rna_mapped]
    prot_genes = [g for g, _ in prot_mapped]
    prot_ids   = [i for _, i in prot_mapped]

    # Filter dataframes to mapped genes only
    rna_values  = rna_df[rna_genes].values
    prot_values = prot_df[prot_genes].values

    # Normalize using paper's method
    print("\nApplying cell-wise min-max normalization...")
    rna_norm  = cellwise_minmax(rna_values)
    prot_norm = cellwise_minmax(prot_values)

    return {
        'rna_values':  rna_norm,
        'rna_ids':     np.array(rna_ids),
        'rna_genes':   rna_genes,
        'prot_values': prot_norm,
        'prot_ids':    np.array(prot_ids),
        'prot_genes':  prot_genes,
        'cell_names':  rna_df.index.tolist(),
        'n_cells':     rna_df.shape[0],
        'n_rna_genes': len(rna_genes),
        'n_proteins':  len(prot_genes),
        # keep mapping stats for the summary file
        'mapping_stats': {
            'rna_total':           rna_orig_cols,
            'rna_dropped_nan':     len(rna_drop_cols),
            'rna_filled_nan':      len(rna_fill_cols),
            'rna_after_nan':       len(rna_df.columns),
            'rna_mapped':          len(rna_mapped),
            'rna_unmapped':        len(rna_unmapped),
            'prot_total':          prot_orig_cols,
            'prot_dropped_nan':    len(prot_drop_cols),
            'prot_filled_nan':     len(prot_fill_cols),
            'prot_after_nan':      len(prot_df.columns),
            'prot_mapped':         len(prot_mapped),
            'prot_unmapped':       len(prot_unmapped),
            'prot_unmapped_names': prot_unmapped,
            'nan_threshold':       NAN_THRESHOLD,
        },
    }

## 6. Model Inference

In [ ]:
def prepare_batch(rna_values, rna_ids, prot_ids, enc_max_len, dec_max_len, device):
    """
    Prepare a batch for model inference.
    Handles padding/truncation for encoder and decoder sequences.
    """
    batch_size = rna_values.shape[0]
    n_rna  = rna_values.shape[1]
    n_prot = len(prot_ids)

    # Encoder (RNA) – truncate or pad to enc_max_len
    if n_rna >= enc_max_len:
        rna_val  = rna_values[:, :enc_max_len]
        rna_id   = np.tile(rna_ids[:enc_max_len], (batch_size, 1))
        rna_mask = np.ones((batch_size, enc_max_len), dtype=bool)
    else:
        rna_val = np.zeros((batch_size, enc_max_len), dtype=np.float32)
        rna_val[:, :n_rna] = rna_values
        rna_id = np.zeros((batch_size, enc_max_len), dtype=np.int64)
        rna_id[:, :n_rna] = rna_ids
        rna_mask = np.zeros((batch_size, enc_max_len), dtype=bool)
        rna_mask[:, :n_rna] = True

    # Decoder (Protein) – pad to dec_max_len (should be <= 1000)
    prot_val  = np.zeros((batch_size, dec_max_len), dtype=np.float32)
    prot_id   = np.zeros((batch_size, dec_max_len), dtype=np.int64)
    prot_mask = np.zeros((batch_size, dec_max_len), dtype=bool)

    prot_id[:, :n_prot]   = prot_ids
    prot_mask[:, :n_prot] = True

    # Convert to tensors
    rna_val_t  = torch.tensor(rna_val,  dtype=torch.float32).to(device)
    rna_id_t   = torch.tensor(rna_id,   dtype=torch.long).to(device)
    rna_mask_t = torch.tensor(rna_mask,  dtype=torch.bool).to(device)
    prot_id_t  = torch.tensor(prot_id,  dtype=torch.long).to(device)
    prot_mask_t = torch.tensor(prot_mask, dtype=torch.bool).to(device)

    return rna_val_t, rna_id_t, rna_mask_t, prot_id_t, prot_mask_t, n_prot


def run_inference(model, data, config, device):
    """
    Run inference on all cells, chunking proteins as needed.
    Decoder max is 1000, so we chunk and concatenate results.
    """
    model.eval()

    n_cells    = data['n_cells']
    n_proteins = data['n_proteins']
    dec_max    = config['dec_max_seq_len']
    batch_size = config['batch_size']

    # Calculate protein chunks
    n_chunks = int(np.ceil(n_proteins / dec_max))
    print(f"\nProtein chunking: {n_proteins} proteins -> {n_chunks} chunks of <={dec_max}")

    # Storage for all predictions
    all_predictions = np.zeros((n_cells, n_proteins), dtype=np.float32)

    # Process each protein chunk
    for chunk_idx in range(n_chunks):
        start_prot = chunk_idx * dec_max
        end_prot   = min((chunk_idx + 1) * dec_max, n_proteins)
        chunk_prot_ids = data['prot_ids'][start_prot:end_prot]
        chunk_size = end_prot - start_prot

        print(f"  Chunk {chunk_idx+1}/{n_chunks}: proteins {start_prot}-{end_prot} ({chunk_size} proteins)")

        chunk_predictions = []

        # Process in batches
        for batch_start in tqdm(range(0, n_cells, batch_size), desc=f"  Chunk {chunk_idx+1}"):
            batch_end = min(batch_start + batch_size, n_cells)
            batch_rna = data['rna_values'][batch_start:batch_end]

            # Prepare batch
            rna_val, rna_id, rna_mask, prot_id, prot_mask, n_prot = prepare_batch(
                batch_rna, data['rna_ids'], chunk_prot_ids,
                config['enc_max_seq_len'], dec_max, device
            )

            # Forward pass
            with torch.no_grad():
                _, y_hat = model(rna_val, rna_id, prot_id,
                                 enc_mask=rna_mask, dec_mask=prot_mask)
                y_hat = torch.squeeze(y_hat)

                # Handle single sample case
                if len(y_hat.shape) == 1:
                    y_hat = y_hat.unsqueeze(0)

                # Extract only the valid protein predictions (not padding)
                y_hat_valid = y_hat[:, :chunk_size].cpu().numpy()
                chunk_predictions.append(y_hat_valid)

        # Concatenate batch predictions for this chunk
        chunk_predictions = np.vstack(chunk_predictions)
        all_predictions[:, start_prot:end_prot] = chunk_predictions

    return all_predictions

## 7. Metrics

In [ ]:
def compute_metrics(y_true, y_pred):
    """
    Per-cell (sample-wise) metrics, then averaged:
    cosine similarity, PCC, MSE, MAE.
    """
    n_cells = y_true.shape[0]

    cosines, pccs, mses, maes = [], [], [], []

    for i in range(n_cells):
        true_i = y_true[i]
        pred_i = y_pred[i]

        # Cosine similarity
        norm_true = np.linalg.norm(true_i)
        norm_pred = np.linalg.norm(pred_i)
        cos = np.dot(true_i, pred_i) / (norm_true * norm_pred) if (norm_true > 0 and norm_pred > 0) else 0.0
        cosines.append(cos)

        # Pearson correlation
        pcc = pearsonr(true_i, pred_i)[0] if (np.std(true_i) > 0 and np.std(pred_i) > 0) else 0.0
        pccs.append(pcc)

        # MSE / MAE
        mses.append(mean_squared_error(true_i, pred_i))
        maes.append(mean_absolute_error(true_i, pred_i))

    return {
        'cosine': np.array(cosines),
        'pcc':    np.array(pccs),
        'mse':    np.array(mses),
        'mae':    np.array(maes),
    }


def compute_protein_wise_metrics(y_true, y_pred):
    """Compute PCC per protein (across all cells).  Returns 1-D array of PCCs (legacy API)."""
    n_proteins = y_true.shape[1]
    pccs = []
    for j in range(n_proteins):
        true_j = y_true[:, j]
        pred_j = y_pred[:, j]
        pcc = pearsonr(true_j, pred_j)[0] if (np.std(true_j) > 0 and np.std(pred_j) > 0) else 0.0
        pccs.append(pcc)
    return np.array(pccs)


def compute_protein_wise_metrics_full(y_true, y_pred):
    """
    Compute per-protein metrics (across all cells): Cosine, PCC, MSE, R².
    Returns a dict of 1-D arrays, one value per protein.
    """
    from sklearn.metrics import r2_score as _r2_score
    n_proteins = y_true.shape[1]
    cosines, pccs, mses, r2s = [], [], [], []
    for j in range(n_proteins):
        true_j = y_true[:, j]
        pred_j = y_pred[:, j]
        # Cosine similarity
        norm_t = np.linalg.norm(true_j)
        norm_p = np.linalg.norm(pred_j)
        cos = np.dot(true_j, pred_j) / (norm_t * norm_p) if (norm_t > 0 and norm_p > 0) else 0.0
        cosines.append(cos)
        # PCC
        pcc = pearsonr(true_j, pred_j)[0] if (np.std(true_j) > 0 and np.std(pred_j) > 0) else 0.0
        pccs.append(pcc)
        # MSE
        mses.append(float(np.mean((true_j - pred_j) ** 2)))
        # R²
        if np.std(true_j) > 0:
            r2s.append(float(_r2_score(true_j, pred_j)))
        else:
            r2s.append(0.0)
    return {
        'cosine': np.array(cosines),
        'pcc':    np.array(pccs),
        'mse':    np.array(mses),
        'r2':     np.array(r2s),
    }

## 8. Visualization

In [ ]:
def plot_results(protein_metrics, prot_genes, checkpoint_name, output_dir):
    """Generate per-protein visualization plots (cosine, PCC, MSE, R²)."""
    os.makedirs(output_dir, exist_ok=True)

    protein_cosines = protein_metrics['cosine']
    protein_pccs    = protein_metrics['pcc']
    protein_mses    = protein_metrics['mse']
    protein_r2s     = protein_metrics['r2']
    n_proteins      = len(protein_pccs)

    fig, axes = plt.subplots(2, 3, figsize=(18, 11))
    fig.suptitle(f'scTranslator OVERLAP Benchmark – {checkpoint_name}  (Per-Protein)', fontsize=14)

    # 1. Per-Protein Cosine Similarity
    ax = axes[0, 0]
    ax.hist(protein_cosines, bins=50, edgecolor='black', alpha=0.7, color='#4C72B0')
    ax.axvline(protein_cosines.mean(), color='red', linestyle='--',
               label=f'Mean: {protein_cosines.mean():.3f}')
    ax.set_xlabel('Cosine Similarity')
    ax.set_ylabel('Count')
    ax.set_title('Per-Protein Cosine Similarity')
    ax.legend()

    # 2. Per-Protein PCC
    ax = axes[0, 1]
    ax.hist(protein_pccs, bins=50, edgecolor='black', alpha=0.7, color='orange')
    ax.axvline(protein_pccs.mean(), color='red', linestyle='--',
               label=f'Mean: {protein_pccs.mean():.3f}')
    ax.set_xlabel('Pearson Correlation')
    ax.set_ylabel('Count')
    ax.set_title('Per-Protein PCC (across cells)')
    ax.legend()

    # 3. Per-Protein MSE  (symlog x-axis for extreme ranges)
    ax = axes[0, 2]
    mse_mean = protein_mses.mean()
    # Use log-spaced bins for heavy-tailed MSE distributions
    mse_pos = protein_mses[protein_mses > 0]
    if len(mse_pos) > 0 and mse_pos.max() / max(mse_pos.min(), 1e-12) > 100:
        log_bins = np.logspace(np.log10(max(mse_pos.min(), 1e-12)),
                               np.log10(mse_pos.max()), 50)
        ax.hist(mse_pos, bins=log_bins, edgecolor='black', alpha=0.7, color='green')
        ax.set_xscale('symlog', linthresh=max(1, np.median(mse_pos) * 0.01))
    else:
        ax.hist(protein_mses, bins=50, edgecolor='black', alpha=0.7, color='green')
    mse_lbl = f'{mse_mean:.2e}' if abs(mse_mean) >= 1000 else f'{mse_mean:.4f}'
    ax.axvline(mse_mean, color='red', linestyle='--', label=f'Mean: {mse_lbl}')
    ax.set_xlabel('MSE')
    ax.set_ylabel('Count')
    ax.set_title('Per-Protein MSE')
    ax.legend()

    # 4. Per-Protein R²  (symlog x-axis for extreme negative values)
    ax = axes[1, 0]
    r2_mean = protein_r2s.mean()
    r2_median = np.median(protein_r2s)
    # Determine a sensible linthresh from the data
    r2_abs = np.abs(protein_r2s)
    linthresh_r2 = max(1, np.median(r2_abs) * 0.01)
    ax.hist(protein_r2s, bins=50, edgecolor='black', alpha=0.7, color='#8172B2')
    if r2_abs.max() > 1e4:
        ax.set_xscale('symlog', linthresh=linthresh_r2)
    r2_lbl = f'{r2_mean:.2e}' if abs(r2_mean) >= 1000 else f'{r2_mean:.3f}'
    ax.axvline(r2_mean, color='red', linestyle='--', label=f'Mean: {r2_lbl}')
    r2_med_lbl = f'{r2_median:.2e}' if abs(r2_median) >= 1000 else f'{r2_median:.3f}'
    ax.axvline(r2_median, color='blue', linestyle=':', label=f'Median: {r2_med_lbl}')
    ax.set_xlabel('R²')
    ax.set_ylabel('Count')
    ax.set_title('Per-Protein R²')
    ax.legend(fontsize=8)

    # 5. Top/Bottom proteins by PCC
    ax = axes[1, 1]
    sorted_idx = np.argsort(protein_pccs)[::-1]
    top_10    = sorted_idx[:10]
    bottom_10 = sorted_idx[-10:]

    proteins_to_show = list(top_10) + list(bottom_10)
    labels = [prot_genes[i] for i in proteins_to_show]
    values = [protein_pccs[i] for i in proteins_to_show]
    colors = ['green'] * 10 + ['red'] * 10

    ax.barh(range(20), values, color=colors, alpha=0.7)
    ax.set_yticks(range(20))
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel('PCC')
    ax.set_title('Top 10 / Bottom 10 Proteins by PCC')
    ax.axvline(0, color='black', linestyle='-', linewidth=0.5)

    # 6. Summary statistics (per-protein only)
    ax = axes[1, 2]
    ax.axis('off')

    # Helper: format a number with scientific notation when huge
    def _fmt(v, decimals=4):
        return f'{v:.2e}' if abs(v) >= 1e4 else f'{v:.{decimals}f}'

    summary_text = f"""
    Per-Protein Summary ({checkpoint_name})
    ========================================
    n = {n_proteins} proteins

    Cosine Similarity:
      Mean: {protein_cosines.mean():.4f} +/- {protein_cosines.std():.4f}
      Min:  {protein_cosines.min():.4f}   Max: {protein_cosines.max():.4f}

    Pearson Correlation (PCC):
      Mean: {protein_pccs.mean():.4f} +/- {protein_pccs.std():.4f}
      Positive: {(protein_pccs > 0).sum()} ({100*(protein_pccs > 0).mean():.1f}%)
      > 0.3:    {(protein_pccs > 0.3).sum()} ({100*(protein_pccs > 0.3).mean():.1f}%)
      > 0.5:    {(protein_pccs > 0.5).sum()} ({100*(protein_pccs > 0.5).mean():.1f}%)

    MSE:
      Mean: {_fmt(protein_mses.mean(), 6)} +/- {_fmt(protein_mses.std(), 6)}

    R²:
      Mean:   {_fmt(protein_r2s.mean())} +/- {_fmt(protein_r2s.std())}
      Median: {_fmt(np.median(protein_r2s))}
      Min:    {_fmt(protein_r2s.min())}   Max: {_fmt(protein_r2s.max())}
    """
    ax.text(0.05, 0.95, summary_text, transform=ax.transAxes, fontsize=9,
            verticalalignment='top', fontfamily='monospace')

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'benchmark_{checkpoint_name}.png'), dpi=150)
    plt.show()
    print(f"  Saved plot: benchmark_{checkpoint_name}.png")


def plot_scatter_examples(y_true, y_pred, prot_genes, checkpoint_name, output_dir):
    """Plot scatter plots for selected proteins."""
    protein_pccs = compute_protein_wise_metrics(y_true, y_pred)
    sorted_idx   = np.argsort(protein_pccs)[::-1]

    n = len(sorted_idx)
    indices = [
        sorted_idx[0],        # Best
        sorted_idx[n // 4],   # 75th percentile
        sorted_idx[n // 2],   # Median
        sorted_idx[3*n // 4], # 25th percentile
        sorted_idx[-1],       # Worst
    ]

    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    fig.suptitle(f'True vs Predicted Protein Abundance – {checkpoint_name}', fontsize=12)

    for ax, idx in zip(axes, indices):
        true_vals = y_true[:, idx]
        pred_vals = y_pred[:, idx]
        pcc = protein_pccs[idx]

        ax.scatter(true_vals, pred_vals, alpha=0.5, s=10)
        ax.plot([true_vals.min(), true_vals.max()],
                [true_vals.min(), true_vals.max()],
                'r--', linewidth=1)
        ax.set_xlabel('True')
        ax.set_ylabel('Predicted')
        ax.set_title(f'{prot_genes[idx]}\nPCC={pcc:.3f}')

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'scatter_{checkpoint_name}.png'), dpi=150)
    plt.show()
    print(f"  Saved plot: scatter_{checkpoint_name}.png")

## 9. Benchmark Runner (single checkpoint)

In [ ]:
def benchmark_checkpoint(checkpoint_name, checkpoint_path, data, config, device):
    """Run benchmark for a single checkpoint."""
    print(f"\n{'='*60}")
    print(f"Benchmarking: {checkpoint_name}")
    print(f"Checkpoint: {checkpoint_path}")
    print(f"{'='*60}")

    # Load model
    print("\nLoading model...")
    model = _torch_load(checkpoint_path, map_location=device)
    model.eval()
    print(f"  Model loaded on {device}")

    # Run inference
    print("\nRunning inference...")
    predictions = run_inference(model, data, config, device)

    # Dynamic dimension validation (no hardcoded counts)
    expected_cells    = data['n_cells']
    expected_proteins = data['n_proteins']
    if predictions.shape != (expected_cells, expected_proteins):
        raise ValueError(
            f"Shape mismatch: got {predictions.shape}, "
            f"expected ({expected_cells}, {expected_proteins})"
        )

    # Save raw predictions
    os.makedirs(config['output_dir'], exist_ok=True)
    pred_path = os.path.join(config['output_dir'], f'predictions_{checkpoint_name}.npy')
    np.save(pred_path, predictions)
    np.save(os.path.join(config['output_dir'], f'protein_names_{checkpoint_name}.npy'),
            np.array(data['prot_genes']))
    np.save(os.path.join(config['output_dir'], f'protein_ids_{checkpoint_name}.npy'),
            np.array(data['prot_ids']))
    np.save(os.path.join(config['output_dir'], f'cell_names_{checkpoint_name}.npy'),
            np.array(data['cell_names']))
    np.savez(
        os.path.join(config['output_dir'], f'predictions_{checkpoint_name}.npz'),
        predictions=predictions,
        protein_names=np.array(data['prot_genes']),
        protein_ids=np.array(data['prot_ids']),
        cell_names=np.array(data['cell_names']),
    )
    print(f"  Saved predictions: {pred_path}")

    # Compute metrics
    print("\nComputing metrics...")
    metrics         = compute_metrics(data['prot_values'], predictions)
    protein_metrics = compute_protein_wise_metrics_full(data['prot_values'], predictions)
    protein_pccs    = protein_metrics['pcc']  # kept for backward compat

    # Print summary
    print(f"\n--- Results for {checkpoint_name} ---")
    print(f"Per-Protein Cosine: {protein_metrics['cosine'].mean():.4f} +/- {protein_metrics['cosine'].std():.4f}")
    print(f"Per-Protein PCC:    {protein_pccs.mean():.4f} +/- {protein_pccs.std():.4f}")
    print(f"Per-Protein MSE:    {protein_metrics['mse'].mean():.6f} +/- {protein_metrics['mse'].std():.6f}")
    print(f"Per-Protein R²:     {protein_metrics['r2'].mean():.4f} +/- {protein_metrics['r2'].std():.4f}")

    # Generate plots
    print("\nGenerating visualizations...")
    plot_results(protein_metrics, data['prot_genes'],
                 checkpoint_name, config['output_dir'])
    plot_scatter_examples(data['prot_values'], predictions, data['prot_genes'],
                          checkpoint_name, config['output_dir'])

    # Save detailed results
    results = {
        'checkpoint':      checkpoint_name,
        'metrics':         metrics,
        'protein_pccs':    protein_pccs,
        'protein_metrics': protein_metrics,
        'predictions':     predictions,
        'prot_genes':      data['prot_genes'],
        'cell_names':      data['cell_names'],
    }
    return results

---
## 10. Run the Pipeline

Everything below actually **executes** the benchmark.  
The cells above only defined functions and configuration.

### 10.1 Device Setup

In [ ]:
print("=" * 60)
print("scTranslator Benchmarking Pipeline – OVERLAP Dataset")
print("=" * 60)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"\nDevice: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

os.makedirs(CONFIG['output_dir'], exist_ok=True)

### 10.2 Prepare Data

In [ ]:
data = prepare_data(CONFIG)

### 10.3 Run Benchmarks (all checkpoints)

In [ ]:
all_results = {}

for ckpt_name, ckpt_file in CONFIG['checkpoints'].items():
    if os.path.exists(ckpt_file):
        results = benchmark_checkpoint(ckpt_name, ckpt_file, data, CONFIG, device)
        all_results[ckpt_name] = results

        # Save individual results
        np.savez(
            os.path.join(CONFIG['output_dir'], f'results_{ckpt_name}.npz'),
            predictions=results['predictions'],
            cosine=results['metrics']['cosine'],
            pcc=results['metrics']['pcc'],
            mse=results['metrics']['mse'],
            mae=results['metrics']['mae'],
            protein_pccs=results['protein_pccs'],
        )

        # Clear GPU memory between checkpoints
        if device == 'cuda':
            torch.cuda.empty_cache()
    else:
        print(f"\nWarning: Checkpoint not found: {ckpt_file}")

### 10.4 Checkpoint Comparison Plot

### 10.5 Final Summary & Save

In [ ]:
print("\n" + "=" * 60)
print("FINAL SUMMARY")
print("=" * 60)

ms = data['mapping_stats']

summary_lines = []
for ckpt_name, results in all_results.items():
    m  = results['metrics']
    pp = results['protein_pccs']
    line = (f"{ckpt_name}: Cosine={m['cosine'].mean():.4f}, "
            f"PCC={m['pcc'].mean():.4f}, MSE={m['mse'].mean():.6f}, "
            f"Protein-PCC={pp.mean():.4f}")
    print(line)
    summary_lines.append(line)

with open(os.path.join(CONFIG['output_dir'], 'summary.txt'), 'w') as f:
    f.write("scTranslator Benchmark Summary – iTRAQ Dataset\n")
    f.write("=" * 60 + "\n\n")
    f.write(f"Raw dataset: {data['n_cells']} samples x "
            f"{ms['prot_total']} proteins, {ms['rna_total']} RNA genes\n\n")

    f.write(f"Step 1: NaN Handling (threshold: >{ms['nan_threshold']*100:.0f}% NaN -> drop, "
            f"else fill with 1e-8)\n")
    f.write("-" * 50 + "\n")
    f.write(f"  RNA genes dropped (>{ms['nan_threshold']*100:.0f}% NaN): {ms['rna_dropped_nan']} / {ms['rna_total']} "
            f"({100*ms['rna_dropped_nan']/ms['rna_total']:.1f}%)\n")
    f.write(f"  RNA genes filled  (<=50% NaN):  {ms['rna_filled_nan']} / {ms['rna_total']} "
            f"({100*ms['rna_filled_nan']/ms['rna_total']:.1f}%)\n")
    f.write(f"  Proteins dropped  (>{ms['nan_threshold']*100:.0f}% NaN): {ms['prot_dropped_nan']} / {ms['prot_total']} "
            f"({100*ms['prot_dropped_nan']/ms['prot_total']:.1f}%)\n")
    f.write(f"  Proteins filled   (<=50% NaN):  {ms['prot_filled_nan']} / {ms['prot_total']} "
            f"({100*ms['prot_filled_nan']/ms['prot_total']:.1f}%)\n")
    f.write(f"  RNA genes remaining:          {ms['rna_after_nan']}\n")
    f.write(f"  Proteins remaining:           {ms['prot_after_nan']}\n\n")

    f.write("Step 2: scTranslator Vocabulary Mapping\n")
    f.write("-" * 50 + "\n")
    f.write(f"  RNA genes mapped:             {ms['rna_mapped']} / {ms['rna_after_nan']} "
            f"({100*ms['rna_mapped']/ms['rna_after_nan']:.1f}%)\n")
    f.write(f"  RNA genes unmapped:           {ms['rna_unmapped']}\n")
    f.write(f"  Proteins mapped:              {ms['prot_mapped']} / {ms['prot_after_nan']} "
            f"({100*ms['prot_mapped']/ms['prot_after_nan']:.1f}%)\n")
    f.write(f"  Proteins unmapped:            {ms['prot_unmapped']}\n")
    f.write(f"  Unmapped protein names:       {ms['prot_unmapped_names']}\n\n")

    f.write("Total Attrition\n")
    f.write("-" * 50 + "\n")
    f.write(f"  RNA:  {ms['rna_total']} -> {data['n_rna_genes']}  "
            f"(dropped {ms['rna_dropped_nan']} NaN + {ms['rna_unmapped']} unmapped, "
            f"{ms['rna_filled_nan']} cols NaN-filled)\n")
    f.write(f"  Prot: {ms['prot_total']} -> {data['n_proteins']}  "
            f"(dropped {ms['prot_dropped_nan']} NaN + {ms['prot_unmapped']} unmapped, "
            f"{ms['prot_filled_nan']} cols NaN-filled)\n\n")

    f.write(f"Final input: {data['n_cells']} samples x "
            f"{data['n_proteins']} proteins, {data['n_rna_genes']} RNA genes\n\n")
    f.write("Benchmark Results\n")
    f.write("-" * 50 + "\n")
    for line in summary_lines:
        f.write(line + "\n")

print(f"\nResults saved to: {CONFIG['output_dir']}")
print("Done!")

---
## 11. Tissue-Level Analysis

Aggregate per-cell metrics by tissue and visualise results for the **three focus tissues**:
`PANCREA2021`, `HEADNECK_2021`, `RENAL_2019`.

For each tissue we:
1. Average the per-cell metrics (Cosine, PCC, MSE, MAE, R squared) across all samples belonging to that tissue.
2. Re-compute protein-wise PCC using only the samples from that tissue.
3. Generate grouped bar charts, box plots, and protein PCC histograms.

In [ ]:
from collections import Counter

# ── Helper: extract tissue from sample name ────────────────────────────────
def get_tissue(sample_name):
    """
    Extract tissue identifier from sample name.
    e.g.  'BRAIN_2021_3'  ->  'BRAIN_2021'
          'BREAST2020_12' ->  'BREAST2020'
    """
    parts = sample_name.rsplit('_', 1)
    if len(parts) == 2 and parts[1].isdigit():
        return parts[0]
    return sample_name

# ── Focus tissues & display names ──────────────────────────────────────────
FOCUS_TISSUES = ['PANCREA2021', 'HEADNECK_2021', 'RENAL_2019']
TISSUE_DISPLAY = {
    'PANCREA2021':   'Pancreas',
    'HEADNECK_2021': 'Head & Neck',
    'RENAL_2019':    'Renal',
}

# ── Derive tissue labels for every sample ──────────────────────────────────
cell_names = data['cell_names']
tissues = np.array([get_tissue(n) for n in cell_names])

tissue_counts = Counter(tissues)
print("Sample counts for focus tissues:")
for t in FOCUS_TISSUES:
    print(f"  {TISSUE_DISPLAY[t]:15s} ({t}): {tissue_counts.get(t, 0)} samples")

# ── Compute tissue-level metrics for each checkpoint ───────────────────────
tissue_summary = {}  # {ckpt: {tissue: {metric: value, ...}}}

for ckpt_name, results in all_results.items():
    tissue_summary[ckpt_name] = {}
    metrics = results['metrics']

    for tissue in FOCUS_TISSUES:
        mask = tissues == tissue
        n = int(mask.sum())
        if n == 0:
            print(f"  WARNING: no samples for {tissue}")
            continue

        ts = {
            'n_samples':   n,
            'cosine_mean': float(metrics['cosine'][mask].mean()),
            'cosine_std':  float(metrics['cosine'][mask].std()),
            'pcc_mean':    float(metrics['pcc'][mask].mean()),
            'pcc_std':     float(metrics['pcc'][mask].std()),
            'mse_mean':    float(metrics['mse'][mask].mean()),
            'mse_std':     float(metrics['mse'][mask].std()),
            'mae_mean':    float(metrics['mae'][mask].mean()),
            'mae_std':     float(metrics['mae'][mask].std()),
        }

        # Tissue-specific protein-wise PCC
        y_true_tissue = data['prot_values'][mask]
        y_pred_tissue = results['predictions'][mask]
        prot_pccs = compute_protein_wise_metrics(y_true_tissue, y_pred_tissue)
        ts['prot_pcc_mean'] = float(prot_pccs.mean())
        ts['prot_pcc_std']  = float(prot_pccs.std())
        ts['prot_pccs']     = prot_pccs

        # Tissue-specific per-cell (per-sample) R²
        from sklearn.metrics import r2_score as _r2
        sample_r2s = []
        for si in np.where(mask)[0]:
            t_i = data['prot_values'][si]
            p_i = results['predictions'][si]
            if np.std(t_i) > 0:
                sample_r2s.append(float(_r2(t_i, p_i)))
            else:
                sample_r2s.append(0.0)
        sample_r2s = np.array(sample_r2s)
        ts['r2_cell_mean'] = float(sample_r2s.mean())
        ts['r2_cell_std']  = float(sample_r2s.std())
        ts['r2_cell_vals'] = sample_r2s

        # Tissue-specific per-protein R²
        n_prot = y_true_tissue.shape[1]
        prot_r2s = []
        for j in range(n_prot):
            true_j = y_true_tissue[:, j]
            pred_j = y_pred_tissue[:, j]
            if np.std(true_j) > 0:
                prot_r2s.append(float(_r2(true_j, pred_j)))
            else:
                prot_r2s.append(0.0)
        prot_r2s = np.array(prot_r2s)
        ts['r2_prot_mean'] = float(prot_r2s.mean())
        ts['r2_prot_std']  = float(prot_r2s.std())
        ts['r2_prot_vals'] = prot_r2s

        tissue_summary[ckpt_name][tissue] = ts

# ── Print summary table ───────────────────────────────────────────────────
print("\n" + "=" * 100)
print("TISSUE-LEVEL ANALYSIS  —  Focus Tissues")
print("=" * 100)
for ckpt_name in all_results:
    print(f"\n  ── {ckpt_name} {'─' * (80 - len(ckpt_name))}")
    print(f"  {'Tissue':15s} {'n':>5s}  {'Cosine':>14s}  {'PCC':>14s}  "
          f"{'MSE':>16s}  {'MAE':>14s}  {'Prot-PCC':>14s}  {'R²(cell)':>14s}  {'R²(prot)':>14s}")
    print(f"  {'-'*15} {'-'*5}  {'-'*14}  {'-'*14}  {'-'*16}  {'-'*14}  {'-'*14}  {'-'*14}  {'-'*14}")
    for tissue in FOCUS_TISSUES:
        if tissue not in tissue_summary[ckpt_name]:
            continue
        ts = tissue_summary[ckpt_name][tissue]
        print(f"  {TISSUE_DISPLAY[tissue]:15s} {ts['n_samples']:5d}  "
              f"{ts['cosine_mean']:+.4f}±{ts['cosine_std']:.4f}  "
              f"{ts['pcc_mean']:+.4f}±{ts['pcc_std']:.4f}  "
              f"{ts['mse_mean']:10.4f}±{ts['mse_std']:.4f}  "
              f"{ts['mae_mean']:+.4f}±{ts['mae_std']:.4f}  "
              f"{ts['prot_pcc_mean']:+.4f}±{ts['prot_pcc_std']:.4f}  "
              f"{ts['r2_cell_mean']:+.4f}±{ts['r2_cell_std']:.4f}  "
              f"{ts['r2_prot_mean']:+.4f}±{ts['r2_prot_std']:.4f}")

# ═══════════════════════════════════════════════════════════════════════════
# FIGURE 1 — Grouped bar chart: 5 metrics × 3 tissues × N checkpoints
# ═══════════════════════════════════════════════════════════════════════════
checkpoints = list(all_results.keys())
n_ckpt = len(checkpoints)

metric_defs = [
    ('cosine_mean', 'cosine_std', 'Cosine Similarity'),
    ('pcc_mean',    'pcc_std',    'Sample-wise PCC'),
    ('mse_mean',    'mse_std',    'MSE'),
    ('mae_mean',    'mae_std',    'MAE'),
    ('prot_pcc_mean', 'prot_pcc_std', 'Protein-wise PCC'),
    ('r2_cell_mean', 'r2_cell_std', 'R² (Per-Cell)'),
    ('r2_prot_mean', 'r2_prot_std', 'R² (Per-Protein)'),
]

ckpt_colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2'][:n_ckpt]

n_metric_plots = len(metric_defs)
n_cols = 3
n_rows = (n_metric_plots + n_cols - 1) // n_cols   # ceiling division

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5 * n_rows))
fig.suptitle('Tissue-Level Benchmark  —  Focus Tissues', fontsize=16, fontweight='bold')

x = np.arange(len(FOCUS_TISSUES))
bar_width = 0.8 / max(n_ckpt, 1)

for ax_idx, (m_key, s_key, title) in enumerate(metric_defs):
    ax = axes.flat[ax_idx]

    # Check if this metric has extreme values → use symlog & skip annotations/errorbars
    all_vals = [tissue_summary[ckpt].get(t, {}).get(m_key, 0)
                for ckpt in checkpoints for t in FOCUS_TISSUES]
    extreme = any(abs(v) > 1e4 for v in all_vals)

    for j, ckpt in enumerate(checkpoints):
        means = [tissue_summary[ckpt].get(t, {}).get(m_key, 0) for t in FOCUS_TISSUES]
        stds  = [tissue_summary[ckpt].get(t, {}).get(s_key, 0) for t in FOCUS_TISSUES]
        offset = (j - n_ckpt / 2 + 0.5) * bar_width
        bars = ax.bar(x + offset, means, bar_width,
                      yerr=None if extreme else stds,
                      label=ckpt, color=ckpt_colors[j], alpha=0.85,
                      capsize=4, edgecolor='white', linewidth=0.5)

    if extreme:
        ax.set_yscale('symlog', linthresh=1)
    ax.set_xticks(x)
    ax.set_xticklabels([TISSUE_DISPLAY[t] for t in FOCUS_TISSUES], fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3, linewidth=0.5)

# Hide any unused subplots
for unused_idx in range(n_metric_plots, n_rows * n_cols):
    axes.flat[unused_idx].axis('off')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(os.path.join(CONFIG['output_dir'], 'tissue_level_summary.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("  Saved: tissue_level_summary.png")

# ═══════════════════════════════════════════════════════════════════════════
# FIGURE 2 — Box plots: per-sample PCC distribution per tissue, per ckpt
# ═══════════════════════════════════════════════════════════════════════════
tissue_colors = ['#E8A838', '#8CD17D', '#B6992D']

fig, axes = plt.subplots(1, n_ckpt, figsize=(5.5 * n_ckpt, 5), sharey=True)
if n_ckpt == 1:
    axes = [axes]
fig.suptitle('Per-Sample PCC Distribution by Tissue', fontsize=14, fontweight='bold')

for ax, ckpt in zip(axes, checkpoints):
    box_data = []
    labels   = []
    for tissue in FOCUS_TISSUES:
        mask = tissues == tissue
        box_data.append(all_results[ckpt]['metrics']['pcc'][mask])
        labels.append(TISSUE_DISPLAY[tissue])

    bp = ax.boxplot(box_data, labels=labels, patch_artist=True, widths=0.55,
                    medianprops=dict(color='black', linewidth=1.5))
    for patch, color in zip(bp['boxes'], tissue_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)

    ax.set_title(ckpt, fontsize=12, fontweight='bold')
    if ax is axes[0]:
        ax.set_ylabel('Pearson Correlation')
    ax.grid(axis='y', alpha=0.3, linewidth=0.5)
    ax.axhline(0, color='grey', linewidth=0.7, linestyle='--')

plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig(os.path.join(CONFIG['output_dir'], 'tissue_pcc_boxplots.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("  Saved: tissue_pcc_boxplots.png")

# ═══════════════════════════════════════════════════════════════════════════
# FIGURE 3 — Overlaid histograms: protein-wise PCC per tissue, per ckpt
# ═══════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, n_ckpt, figsize=(5.5 * n_ckpt, 5), sharey=True)
if n_ckpt == 1:
    axes = [axes]
fig.suptitle('Protein-wise PCC Distribution by Tissue', fontsize=14, fontweight='bold')

for ax, ckpt in zip(axes, checkpoints):
    for k, tissue in enumerate(FOCUS_TISSUES):
        if tissue not in tissue_summary[ckpt]:
            continue
        ppccs = tissue_summary[ckpt][tissue]['prot_pccs']
        ax.hist(ppccs, bins=40, alpha=0.45, label=TISSUE_DISPLAY[tissue],
                color=tissue_colors[k], edgecolor='white', linewidth=0.3)
        ax.axvline(ppccs.mean(), color=tissue_colors[k],
                   linestyle='--', linewidth=1.8,
                   label=f'{TISSUE_DISPLAY[tissue]} mean={ppccs.mean():.3f}')
    ax.set_title(ckpt, fontsize=12, fontweight='bold')
    ax.set_xlabel('Protein-wise PCC')
    if ax is axes[0]:
        ax.set_ylabel('Count')
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(axis='y', alpha=0.3, linewidth=0.5)

plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig(os.path.join(CONFIG['output_dir'], 'tissue_protein_pcc_hist.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("  Saved: tissue_protein_pcc_hist.png")

# ═══════════════════════════════════════════════════════════════════════════
# FIGURE 4 — Heatmap-style summary table (Cosine & PCC) for quick glance
# ═══════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(max(8, 3 * n_ckpt), 4))
ax.set_title('Tissue × Checkpoint — Mean Sample-wise PCC', fontsize=13, fontweight='bold')

table_data = np.zeros((len(FOCUS_TISSUES), n_ckpt))
for j, ckpt in enumerate(checkpoints):
    for i, tissue in enumerate(FOCUS_TISSUES):
        table_data[i, j] = tissue_summary[ckpt].get(tissue, {}).get('pcc_mean', np.nan)

im = ax.imshow(table_data, cmap='RdYlGn', aspect='auto')
ax.set_xticks(range(n_ckpt))
ax.set_xticklabels(checkpoints, fontsize=11)
ax.set_yticks(range(len(FOCUS_TISSUES)))
ax.set_yticklabels([TISSUE_DISPLAY[t] for t in FOCUS_TISSUES], fontsize=11)

# Annotate each cell
for i in range(len(FOCUS_TISSUES)):
    for j in range(n_ckpt):
        val = table_data[i, j]
        color = 'white' if abs(val) > 0.3 else 'black'
        ax.text(j, i, f'{val:.4f}', ha='center', va='center',
                fontsize=12, fontweight='bold', color=color)

plt.colorbar(im, ax=ax, label='PCC')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'tissue_pcc_heatmap.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("  Saved: tissue_pcc_heatmap.png")

# ═══════════════════════════════════════════════════════════════════════════
# Save tissue-level results to CSV
# ═══════════════════════════════════════════════════════════════════════════
rows = []
for ckpt in checkpoints:
    for tissue in FOCUS_TISSUES:
        if tissue not in tissue_summary[ckpt]:
            continue
        ts = tissue_summary[ckpt][tissue]
        rows.append({
            'Checkpoint':  ckpt,
            'Tissue':      TISSUE_DISPLAY[tissue],
            'Tissue_ID':   tissue,
            'N_samples':   ts['n_samples'],
            'Cosine_mean': round(ts['cosine_mean'], 6),
            'Cosine_std':  round(ts['cosine_std'], 6),
            'PCC_mean':    round(ts['pcc_mean'], 6),
            'PCC_std':     round(ts['pcc_std'], 6),
            'MSE_mean':    round(ts['mse_mean'], 6),
            'MSE_std':     round(ts['mse_std'], 6),
            'MAE_mean':    round(ts['mae_mean'], 6),
            'MAE_std':     round(ts['mae_std'], 6),
            'ProtPCC_mean':   round(ts['prot_pcc_mean'], 6),
            'ProtPCC_std':    round(ts['prot_pcc_std'], 6),
            'R2_cell_mean':   round(ts['r2_cell_mean'], 6),
            'R2_cell_std':    round(ts['r2_cell_std'], 6),
            'R2_prot_mean':   round(ts['r2_prot_mean'], 6),
            'R2_prot_std':    round(ts['r2_prot_std'], 6),
        })

tissue_df = pd.DataFrame(rows)
tissue_csv_path = os.path.join(CONFIG['output_dir'], 'tissue_level_metrics.csv')
tissue_df.to_csv(tissue_csv_path, index=False)

print(f"\nTissue-level metrics saved to: {tissue_csv_path}")
print("\n" + tissue_df.to_string(index=False))

---
## 12. All-Tissue Analysis

Extends the analysis to **every tissue** in the dataset.  
Because 12 tissues × 3 checkpoints generates a lot of numbers, we lean on:

| Visualisation | What it shows |
|---|---|
| **Annotated heatmaps** | Tissue × Checkpoint matrix for each metric — colour lets you spot patterns at a glance |
| **Sorted horizontal bar charts** | All tissues ranked by mean PCC; the three focus tissues are highlighted in colour |
| **Combined dot-plot** | Cosine & PCC side-by-side with error bars for every tissue |

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 12. ALL-TISSUE ANALYSIS
# ═══════════════════════════════════════════════════════════════════════════════
import matplotlib.ticker as mticker

# ── Discover every tissue in the dataset ───────────────────────────────────
ALL_TISSUES = sorted(set(tissues))
print(f"Found {len(ALL_TISSUES)} tissues:")
for t in ALL_TISSUES:
    print(f"  {t:25s}  ({tissue_counts[t]} samples)")

# Friendly display names (auto-generate from IDs)
def _display(t):
    """Turn e.g. 'HEADNECK_2021' into 'Head & Neck 2021'."""
    return (t.replace('_', ' ')
             .replace('HEADNECK', 'Head & Neck')
             .replace('PANCREA', 'Pancreas ')
             .replace('PEDBRAIN', 'Ped. Brain')
             .title())

ALL_DISPLAY = {t: _display(t) for t in ALL_TISSUES}

# ── Compute tissue-level metrics for ALL tissues & all checkpoints ─────────
all_tissue_summary = {}  # {ckpt: {tissue: {...}}}

for ckpt_name, results in all_results.items():
    all_tissue_summary[ckpt_name] = {}
    metrics = results['metrics']

    for tissue in ALL_TISSUES:
        mask = tissues == tissue
        n = int(mask.sum())
        if n == 0:
            continue

        ts = {
            'n_samples':   n,
            'cosine_mean': float(metrics['cosine'][mask].mean()),
            'cosine_std':  float(metrics['cosine'][mask].std()),
            'pcc_mean':    float(metrics['pcc'][mask].mean()),
            'pcc_std':     float(metrics['pcc'][mask].std()),
            'mse_mean':    float(metrics['mse'][mask].mean()),
            'mse_std':     float(metrics['mse'][mask].std()),
            'mae_mean':    float(metrics['mae'][mask].mean()),
            'mae_std':     float(metrics['mae'][mask].std()),
        }

        # tissue-specific protein-wise PCC
        y_true_t = data['prot_values'][mask]
        y_pred_t = results['predictions'][mask]
        ppccs = compute_protein_wise_metrics(y_true_t, y_pred_t)
        ts['prot_pcc_mean'] = float(ppccs.mean())
        ts['prot_pcc_std']  = float(ppccs.std())

        # tissue-specific per-cell (per-sample) R²
        from sklearn.metrics import r2_score as _r2
        sample_r2s = []
        for si in np.where(mask)[0]:
            t_i = data['prot_values'][si]
            p_i = results['predictions'][si]
            if np.std(t_i) > 0:
                sample_r2s.append(float(_r2(t_i, p_i)))
            else:
                sample_r2s.append(0.0)
        sample_r2s = np.array(sample_r2s)
        ts['r2_cell_mean'] = float(sample_r2s.mean())
        ts['r2_cell_std']  = float(sample_r2s.std())

        # tissue-specific per-protein R²
        n_prot = y_true_t.shape[1]
        prot_r2s = []
        for j in range(n_prot):
            true_j = y_true_t[:, j]
            pred_j = y_pred_t[:, j]
            if np.std(true_j) > 0:
                prot_r2s.append(float(_r2(true_j, pred_j)))
            else:
                prot_r2s.append(0.0)
        prot_r2s = np.array(prot_r2s)
        ts['r2_prot_mean'] = float(prot_r2s.mean())
        ts['r2_prot_std']  = float(prot_r2s.std())

        all_tissue_summary[ckpt_name][tissue] = ts

checkpoints = list(all_results.keys())
n_ckpt = len(checkpoints)

# ── Print full table ──────────────────────────────────────────────────────
print("\n" + "=" * 110)
print("ALL-TISSUE ANALYSIS  —  Averaged Per-Sample Metrics")
print("=" * 110)
for ckpt in checkpoints:
    print(f"\n  ── {ckpt} {'─' * (90 - len(ckpt))}")
    print(f"  {'Tissue':25s} {'n':>5s}  {'Cosine':>14s}  {'PCC':>14s}  "
          f"{'MSE':>16s}  {'MAE':>14s}  {'Prot-PCC':>14s}  {'R²(cell)':>14s}  {'R²(prot)':>14s}")
    print(f"  {'-'*25} {'-'*5}  {'-'*14}  {'-'*14}  {'-'*16}  {'-'*14}  {'-'*14}  {'-'*14}  {'-'*14}")
    for tissue in ALL_TISSUES:
        ts = all_tissue_summary[ckpt].get(tissue)
        if ts is None:
            continue
        print(f"  {ALL_DISPLAY[tissue]:25s} {ts['n_samples']:5d}  "
              f"{ts['cosine_mean']:+.4f}±{ts['cosine_std']:.4f}  "
              f"{ts['pcc_mean']:+.4f}±{ts['pcc_std']:.4f}  "
              f"{ts['mse_mean']:10.4f}±{ts['mse_std']:.4f}  "
              f"{ts['mae_mean']:+.4f}±{ts['mae_std']:.4f}  "
              f"{ts['prot_pcc_mean']:+.4f}±{ts['prot_pcc_std']:.4f}  "
              f"{ts['r2_cell_mean']:+.4f}±{ts['r2_cell_std']:.4f}  "
              f"{ts['r2_prot_mean']:+.4f}±{ts['r2_prot_std']:.4f}")

# ═══════════════════════════════════════════════════════════════════════════
# FIGURE A — Annotated heatmaps for 4 key metrics
# ═══════════════════════════════════════════════════════════════════════════
metric_heatmap_defs = [
    ('cosine_mean',   'Cosine Similarity',        'RdYlGn'),
    ('pcc_mean',      'Sample-wise PCC',           'RdYlGn'),
    ('prot_pcc_mean', 'Protein-wise PCC',          'RdYlGn'),
    ('mse_mean',      'MSE (lower is better)',     'RdYlGn_r'),
    ('r2_cell_mean',  'R² (Per-Cell)',             'RdYlGn'),
    ('r2_prot_mean',  'R² (Per-Protein)',          'RdYlGn'),
]

fig, axes = plt.subplots(3, 2, figsize=(max(10, 4 * n_ckpt), 18))
fig.suptitle('All-Tissue Benchmark  —  Heatmaps', fontsize=16, fontweight='bold')

for ax, (m_key, title, cmap) in zip(axes.flat, metric_heatmap_defs):
    mat = np.full((len(ALL_TISSUES), n_ckpt), np.nan)
    for j, ckpt in enumerate(checkpoints):
        for i, tissue in enumerate(ALL_TISSUES):
            ts = all_tissue_summary[ckpt].get(tissue)
            if ts:
                mat[i, j] = ts[m_key]

    im = ax.imshow(mat, cmap=cmap, aspect='auto')
    ax.set_xticks(range(n_ckpt))
    ax.set_xticklabels(checkpoints, fontsize=10)
    ax.set_yticks(range(len(ALL_TISSUES)))
    ax.set_yticklabels([ALL_DISPLAY[t] for t in ALL_TISSUES], fontsize=9)
    ax.set_title(title, fontsize=12, fontweight='bold', pad=8)

    # annotate cells
    for i in range(len(ALL_TISSUES)):
        for j in range(n_ckpt):
            val = mat[i, j]
            if np.isnan(val):
                continue
            vmin, vmax = np.nanmin(mat), np.nanmax(mat)
            mid = (vmin + vmax) / 2
            color = 'white' if abs(val - mid) > (vmax - vmin) * 0.3 else 'black'
            if abs(val) >= 1000:
                fmt = f'{val:.0f}'
            elif abs(val) >= 100:
                fmt = f'{val:.1f}'
            else:
                fmt = f'{val:.4f}'
            ax.text(j, i, fmt, ha='center', va='center',
                    fontsize=8, fontweight='bold', color=color)

    plt.colorbar(im, ax=ax, shrink=0.85)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(os.path.join(CONFIG['output_dir'], 'all_tissue_heatmaps.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("  Saved: all_tissue_heatmaps.png")

# ═══════════════════════════════════════════════════════════════════════════
# FIGURE B — Sorted horizontal bar charts (PCC & Cosine), focus highlighted
# ═══════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(n_ckpt, 2, figsize=(14, max(6, 2.2 * len(ALL_TISSUES))),
                         gridspec_kw={'wspace': 0.35})
if n_ckpt == 1:
    axes = axes[np.newaxis, :]
fig.suptitle('All Tissues Ranked by Mean Metric  (highlighted = focus tissues)',
             fontsize=14, fontweight='bold')

for row, ckpt in enumerate(checkpoints):
    for col, (m_key, label) in enumerate([('pcc_mean', 'Sample-wise PCC'),
                                           ('cosine_mean', 'Cosine Similarity')]):
        ax = axes[row, col]

        # Gather values and sort
        vals = []
        for t in ALL_TISSUES:
            ts = all_tissue_summary[ckpt].get(t)
            v  = ts[m_key] if ts else np.nan
            sd = ts.get(m_key.replace('mean', 'std'), 0) if ts else 0
            vals.append((t, v, sd))
        vals.sort(key=lambda x: x[1])

        y_pos = np.arange(len(vals))
        bar_vals  = [v for _, v, _ in vals]
        bar_errs  = [s for _, _, s in vals]
        bar_names = [ALL_DISPLAY[t] for t, _, _ in vals]
        bar_colors = ['#E8A838' if t in FOCUS_TISSUES else '#aec7e8' for t, _, _ in vals]

        ax.barh(y_pos, bar_vals, xerr=bar_errs, color=bar_colors,
                edgecolor='white', linewidth=0.5, alpha=0.85, capsize=3, height=0.7)
        ax.set_yticks(y_pos)
        ax.set_yticklabels(bar_names, fontsize=9)
        ax.set_xlabel(label)
        ax.set_title(f'{ckpt}  —  {label}', fontsize=11, fontweight='bold')
        ax.axvline(0, color='grey', linewidth=0.6, linestyle='--')
        ax.grid(axis='x', alpha=0.3, linewidth=0.5)

        # Annotate values on bars
        for yp, v in zip(y_pos, bar_vals):
            ax.text(v + 0.002 if v >= 0 else v - 0.002, yp,
                    f'{v:.4f}', va='center',
                    ha='left' if v >= 0 else 'right', fontsize=7)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(os.path.join(CONFIG['output_dir'], 'all_tissue_ranked_bars.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("  Saved: all_tissue_ranked_bars.png")

# ═══════════════════════════════════════════════════════════════════════════
# FIGURE C — Dot-plot: Cosine & PCC with error bars for every tissue
#             One row per checkpoint, two columns (Cosine, PCC)
# ═══════════════════════════════════════════════════════════════════════════
# Sort tissues by average PCC across all checkpoints for consistent ordering
avg_pcc_order = sorted(
    ALL_TISSUES,
    key=lambda t: np.mean([all_tissue_summary[c].get(t, {}).get('pcc_mean', -999)
                           for c in checkpoints]),
    reverse=True)

fig, axes = plt.subplots(1, n_ckpt, figsize=(6 * n_ckpt, max(6, 0.55 * len(ALL_TISSUES))),
                         sharey=True)
if n_ckpt == 1:
    axes = [axes]
fig.suptitle('Sample-wise PCC by Tissue  (dot = mean, whisker = ±1 std)',
             fontsize=14, fontweight='bold')

y_pos = np.arange(len(avg_pcc_order))

for ax, ckpt in zip(axes, checkpoints):
    means = []
    stds  = []
    colors = []
    for t in avg_pcc_order:
        ts = all_tissue_summary[ckpt].get(t, {})
        means.append(ts.get('pcc_mean', np.nan))
        stds.append(ts.get('pcc_std', 0))
        colors.append('#E8A838' if t in FOCUS_TISSUES else '#4C72B0')

    ax.errorbar(means, y_pos, xerr=stds, fmt='o', markersize=6,
                elinewidth=1.2, capsize=3, capthick=1,
                color='#4C72B0', markerfacecolor='none', zorder=2)
    # Overlay filled dots for focus tissues
    for i, t in enumerate(avg_pcc_order):
        if t in FOCUS_TISSUES:
            ax.plot(means[i], y_pos[i], 'o', markersize=8,
                    color='#E8A838', markeredgecolor='#d48000', zorder=3)

    ax.set_yticks(y_pos)
    ax.set_yticklabels([ALL_DISPLAY[t] for t in avg_pcc_order], fontsize=9)
    ax.set_xlabel('Sample-wise PCC')
    ax.set_title(ckpt, fontsize=12, fontweight='bold')
    ax.axvline(0, color='grey', linewidth=0.6, linestyle='--')
    ax.grid(axis='x', alpha=0.3, linewidth=0.5)

# Add a manual legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#E8A838',
           markeredgecolor='#d48000', markersize=8, label='Focus tissue'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='none',
           markeredgecolor='#4C72B0', markersize=7, label='Other tissue'),
]
axes[-1].legend(handles=legend_elements, fontsize=9, loc='lower right')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(os.path.join(CONFIG['output_dir'], 'all_tissue_dotplot.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("  Saved: all_tissue_dotplot.png")

# ═══════════════════════════════════════════════════════════════════════════
# FIGURE D — Grouped box-plot: per-sample PCC distributions, all tissues
# ═══════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(n_ckpt, 1,
                         figsize=(max(12, 1.3 * len(ALL_TISSUES)), 4.5 * n_ckpt))
if n_ckpt == 1:
    axes = [axes]
fig.suptitle('Per-Sample PCC Distributions  —  All Tissues',
             fontsize=14, fontweight='bold')

for ax, ckpt in zip(axes, checkpoints):
    box_data = []
    labels   = []
    face_colors = []
    for t in avg_pcc_order:
        mask = tissues == t
        box_data.append(all_results[ckpt]['metrics']['pcc'][mask])
        labels.append(ALL_DISPLAY[t])
        face_colors.append('#E8A838' if t in FOCUS_TISSUES else '#aec7e8')

    bp = ax.boxplot(box_data, patch_artist=True, widths=0.6,
                    medianprops=dict(color='black', linewidth=1.5))
    for patch, fc in zip(bp['boxes'], face_colors):
        patch.set_facecolor(fc)
        patch.set_alpha(0.75)

    ax.set_title(ckpt, fontsize=12, fontweight='bold')
    ax.set_ylabel('Pearson Correlation')
    ax.set_xticks(range(1, len(labels) + 1))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
    ax.axhline(0, color='grey', linewidth=0.6, linestyle='--')
    ax.grid(axis='y', alpha=0.3, linewidth=0.5)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(os.path.join(CONFIG['output_dir'], 'all_tissue_boxplots.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("  Saved: all_tissue_boxplots.png")

# ═══════════════════════════════════════════════════════════════════════════
# Save ALL-tissue results to CSV
# ═══════════════════════════════════════════════════════════════════════════
rows_all = []
for ckpt in checkpoints:
    for tissue in ALL_TISSUES:
        ts = all_tissue_summary[ckpt].get(tissue)
        if ts is None:
            continue
        rows_all.append({
            'Checkpoint':   ckpt,
            'Tissue':       ALL_DISPLAY[tissue],
            'Tissue_ID':    tissue,
            'N_samples':    ts['n_samples'],
            'Cosine_mean':  round(ts['cosine_mean'], 6),
            'Cosine_std':   round(ts['cosine_std'], 6),
            'PCC_mean':     round(ts['pcc_mean'], 6),
            'PCC_std':      round(ts['pcc_std'], 6),
            'MSE_mean':     round(ts['mse_mean'], 6),
            'MSE_std':      round(ts['mse_std'], 6),
            'MAE_mean':     round(ts['mae_mean'], 6),
            'MAE_std':      round(ts['mae_std'], 6),
            'ProtPCC_mean':   round(ts['prot_pcc_mean'], 6),
            'ProtPCC_std':    round(ts['prot_pcc_std'], 6),
            'R2_cell_mean':   round(ts['r2_cell_mean'], 6),
            'R2_cell_std':    round(ts['r2_cell_std'], 6),
            'R2_prot_mean':   round(ts['r2_prot_mean'], 6),
            'R2_prot_std':    round(ts['r2_prot_std'], 6),
            'Focus':          tissue in FOCUS_TISSUES,
        })

all_tissue_df = pd.DataFrame(rows_all)
all_csv_path = os.path.join(CONFIG['output_dir'], 'all_tissue_metrics.csv')
all_tissue_df.to_csv(all_csv_path, index=False)

print(f"\nAll-tissue metrics saved to: {all_csv_path}")
print("\n" + all_tissue_df.to_string(index=False))


## 13. Per-Protein MSE Table (Excluding Focus Tissues)

Compute the **MSE of each protein averaged across samples**, after removing all
samples from `PANCREA2021`, `HEADNECK_2021`, and `RENAL_2019`.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 13. PER-PROTEIN MSE TABLE  — excluding PANCREA2021, HEADNECK_2021, RENAL_2019
# ═══════════════════════════════════════════════════════════════════════════════

# Ensure tissue labels exist (safe to re-run)
def _get_tissue(name):
    parts = name.rsplit('_', 1)
    return parts[0] if len(parts) == 2 and parts[1].isdigit() else name

_EXCLUDE = {'PANCREA2021', 'HEADNECK_2021', 'RENAL_2019'}

cell_names  = data['cell_names']
_tissues    = np.array([_get_tissue(n) for n in cell_names])
keep_mask   = np.array([t not in _EXCLUDE for t in _tissues])

n_kept      = int(keep_mask.sum())
n_excluded  = len(keep_mask) - n_kept
print(f"Samples kept : {n_kept}  (excluded {n_excluded} from {_EXCLUDE})")

prot_genes  = data['prot_genes']
y_true_filt = data['prot_values'][keep_mask]          # (n_kept, n_proteins)

# ── Build a DataFrame: one row per protein, one MSE column per checkpoint ──
mse_dict = {'Protein': prot_genes}

for ckpt_name, results in all_results.items():
    y_pred_filt = results['predictions'][keep_mask]   # (n_kept, n_proteins)
    # Per-protein MSE = mean over kept samples of (true - pred)^2
    per_protein_mse = np.mean((y_true_filt - y_pred_filt) ** 2, axis=0)
    mse_dict[f'MSE_{ckpt_name}'] = np.round(per_protein_mse, 8)

protein_mse_df = pd.DataFrame(mse_dict)

mse_cols = [c for c in protein_mse_df.columns if c.startswith('MSE_')]
protein_mse_df = protein_mse_df.sort_values(mse_cols[0], ascending=True).reset_index(drop=True)

# ── Save to CSV ───────────────────────────────────────────────────────────
csv_path = os.path.join(CONFIG['output_dir'], 'per_protein_mse_excl_focus.csv')
protein_mse_df.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}")

# ── Display the full table ────────────────────────────────────────────────
pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', '{:.6f}'.format)
print(f"\nPer-Protein MSE  ({n_kept} samples, 3 focus tissues excluded)")
print(f"{'=' * 80}")
print(protein_mse_df.to_string(index=False))

# ── Quick summary stats ──────────────────────────────────────────────────
print(f"\n{'=' * 80}")
print("Summary (across all {0} proteins):".format(len(prot_genes)))
for col in mse_cols:
    vals = protein_mse_df[col]
    print(f"  {col:20s}  mean={vals.mean():.6f}  median={vals.median():.6f}  "
          f"min={vals.min():.6f}  max={vals.max():.6f}")

pd.reset_option('display.max_rows')
pd.reset_option('display.float_format')